# Kolommen wegen: van rollentabel naar vector

In de vorige les rekende `top5` alleen op de omschrijving. Vandaag krijgt elke kolom uit de rollentabel een eigen *blok* in de vector, met een eigen gewicht. We doen het met twee blokken (omschrijving als TF-IDF, genre als one-hot) en kijken wat het gewicht van genre met de top 5 doet. Zet deze tabel over naar je recommender.py notebook of notebook waar je tot nu toe alle code hebt staan.

### Opdracht 1 · De rollentabel

Vul de tabel in voordat je verder leest. Per kolom uit de data hierboven: houd je hem, en zo ja, welke rol krijgt hij in de vector? Doe dit voor alle kolommen in de dataset.

| Kolom | Behouden? j/n, en waarom | Type (TF-IDF · one-hot · hard filter · zachte aanpassing · sleutel), met onderbouwing |
|---|---|---|
| cataloguscode |  |  |
| Titel |  |  |
| Korte omschrijving (omschrijving) |  |  |
| Korte omschrijving (steekwoorden) |  |  |
| trefwoorden |  |  |
| genre |  |  |

*Toelichting op de types.* **TF-IDF**: vrije tekst, wordt een blok woorden. **One-hot**: categorie, één kolom per waarde. **Hard filter**: komt niet in de vector, maar sluit titels uit (bijvoorbeeld een ander niveau). **Zachte aanpassing**: komt niet in de vector, maar schuift de score na het rekenen iets op (bijvoorbeeld: een titel met `Gemiddelde beoordeling` 4 of hoger krijgt 0.05 bij de cosine, zodat van twee bijna even goed passende boeken het best beoordeelde net bovenaan komt). **Sleutel**: identificeert de rij, telt nooit mee.

### De dummy dataset
`raw_books` zijn dezelfde dertig dummy boeken, net zoals in de vorige notebook.

In [33]:
import re
import pandas as pd
import numpy as np

# Dertig boeken en twee tijdschriften, al opgeschoond (kleine letters, één scheidingsteken, geen nepwaarden).
# Dit staat voor jullie eigen opgeschoonde csv; het inlezen daarvan regelen jullie zelf in recommender.py.
raw_books = pd.DataFrame({
    "cataloguscode": [
        "B001", "B002", "B003", "B004", "B005", "B006", "B007", "B008", "B009", "B010", "B011",
        "B012", "B013", "B014", "B015", "B016", "B017", "B018", "B019", "B020", "B021", "B022",
        "B023", "B024", "B025", "B026", "B027", "B028", "B029", "B030", "T001", "T002",
    ],
    "Titel": [
        "de draak onder de school", "leven in de ruimte", "het drakenei", "nieuw in de stad",
        "code rood op de redactie", "vulkanen van dichtbij", "de laatste wolf",
        "mijn broer de robot", "een zomer op terschelling", "bouw je eigen game",
        "de verdwenen schat van kapitein bos", "alles over bijen", "het geheim van de oude molen",
        "eerste hulp bij heimwee", "draken bestaan niet", "op zoek naar mars", "de voetbalschool",
        "koken zonder recept", "het meisje dat niet kon stoppen met rennen",
        "waarom de zee zout is", "sterren kijken voor beginners", "de ruimtereis van juul",
        "slimme machines", "dieren in de winter", "de draak van de brandweer",
        "vriendschap op kamp", "het raadsel van de vuurtoren", "bakken met kinderen",
        "wolven in europa", "de schoolreis naar de bergen", "wetenschap nu, nummer 3",
        "dier & natuur, winter",
    ],
    "Korte omschrijving": [
        "Een meisje ontdekt dat de oude draak onder haar school niet dood is. Alleen zij kan hem horen, en elke nacht wordt hij onrustiger. Samen met haar beste vriend zoekt ze uit hoe ze de draak kan tegenhouden voordat de school instort. Maar niet iedereen op school wil dat de draak verdwijnt.",
        "Hoe eten, slapen en douchen astronauten aan boord van het ruimtestation? Dit boek neemt je mee achter de schermen, van de training op aarde tot de terugkeer na een half jaar in de ruimte. Met veel foto's en antwoorden op vragen van kinderen.",
        "Twee broers vinden in het bos een enorm ei dat warm aanvoelt. Als het uitkomt, staat het hele dorp op zijn kop. De jongens moeten het geheim bewaren, want de burgemeester wil alles wat vreemd is laten opruimen. Een spannend verhaal over broers, een geheim en een verrassing die groter is dan ze dachten.",
        "Na de scheiding van haar ouders verhuist Noor naar de stad. Op een nieuwe school kent ze niemand en alles is anders dan thuis. Langzaam leert ze nieuwe vrienden kennen, maar ze mist haar oude dorp en haar vader. Een verhaal over verhuizen, vriendschap en opnieuw beginnen.",
        "Op de schoolkrant lekt een geheim uit. Wie heeft gepraat, en hoe houd je de redactie bij elkaar? Quinten is hoofdredacteur en moet kiezen tussen de waarheid en zijn vrienden. Een verhaal over school, vriendschap en de macht van een krant.",
        "Waarom barsten vulkanen uit en hoe meten onderzoekers dat? In dit boek lees je wat er onder de grond gebeurt en wat je doet als er een vulkaan in de buurt actief wordt. Met verhalen van vulkanologen die dicht bij de lava werken.",
        "In een dorp in de bergen wordt de laatste wolf gezien. Een jongen wil de wolf redden, maar de boeren willen hem weg. Terwijl de winter dichterbij komt, moet de jongen kiezen tussen zijn dorp en het dier. Een verhaal over vriendschap tussen mens en dier.",
        "Sems ouders kopen een robot die alles beter kan dan hij. Op school, thuis en op het voetbalveld lijkt de robot overal beter in. Totdat de robot iets doet wat niemand had verwacht. Een grappig verhaal over broers, techniek en wat een mens bijzonder maakt.",
        "Drie vrienden werken een zomer lang in een strandtent op Terschelling. Ze ontdekken dat het eiland een geheim bewaart, verstopt tussen de duinen. Wat begint als een vakantiebaantje, wordt een zoektocht naar een oude schat.",
        "Stap voor stap een eigen computerspel maken, van idee tot speelbare versie. Je leert programmeren zonder ervaring, ontwerpt je eigen personages en test je spel met vrienden. In dit boek maak je elke week een nieuw level.",
        "Een oude kaart, een verdwenen schip en een klas die op schoolreis per ongeluk een echte schat vindt. Maar de schat is niet van hen, en iemand anders zoekt er al jaren naar. Een spannend verhaal vol raadsels, geheime gangen en piraten.",
        "Hoe werkt een bijenvolk en waarom verdwijnen bijen? In dit boek lees je wat een koningin doet, hoe bijen honing maken en wat je in je eigen tuin kunt doen om ze te helpen. Met proefjes en weetjes.",
        "De oude molen aan de rand van het dorp draait soms 's nachts. Lotte en haar buurjongen gaan op onderzoek uit en vinden een verborgen kamer. Wie woonde hier vroeger, en waarom mocht niemand het weten? Een spannend verhaal over een geheim dat al honderd jaar bewaard wordt.",
        "Tips voor als je voor het eerst van huis bent, op kamp, op schoolreis of in een gastgezin. Wat doe je als je je verdrietig voelt en wie kun je dan om hulp vragen? Een boek vol praktische tips en verhalen van andere kinderen.",
        "Een jongen die niet in draken gelooft, krijgt er een in zijn tuin. En de draak gelooft niet in jongens. Samen komen ze erachter dat ze meer op elkaar lijken dan ze dachten. Een grappig verhaal over vriendschap en over geloven in dingen die je niet kunt zien.",
        "Wat weten we over de rode planeet? In dit boek lees je hoe de missies naar Mars verlopen, wat robots daar ontdekken en of er ooit mensen zullen wonen. Met foto's uit de ruimte en tekeningen van toekomstige steden.",
        "Op de voetbalschool moet Amir kiezen tussen zijn beste vriend en een plek in het eerste team. Op het veld wordt de strijd steeds harder en op school raakt hij zijn vrienden kwijt. Een verhaal over sport, vriendschap en eerlijk spelen.",
        "Koken op gevoel: wat past bij wat, hoe je proeft en hoe je bijstuurt. Met twintig basisgerechten die je steeds anders kunt maken. In dit boek leer je koken zonder dat je alles hoeft af te wegen.",
        "Fenna rent elke dag harder en verder, tot haar lichaam zegt dat het genoeg is geweest. Haar trainer ziet het niet en haar vriendin durft niets te zeggen. Een eerlijk verhaal over sport, druk en het durven stoppen.",
        "Korte antwoorden op grote vragen over de zee: waarom is het water zout, hoe ontstaan getijden en wat leeft er op de bodem? Met proefjes die je thuis kunt doen en weetjes over stromingen.",
        "Hoe vind je de Grote Beer, en wat zie je door een telescoop? In dit boek leer je de sterren en planeten herkennen, ook als je in de stad woont. Met kaarten van de sterrenhemel en tips om zelf de ruimte te bekijken.",
        "Juul wil astronaut worden en mag meedoen aan een training voor kinderen. Maar tijdens de proef gaat er iets mis in het ruimtestation. Een spannend verhaal over durf, samenwerken en dromen van een reis naar de sterren.",
        "Wat kan een robot en wat kan hij nog niet? In dit boek lees je hoe computers leren, hoe robots worden gebouwd en wat dat betekent voor de toekomst. Met opdrachten om zelf een eenvoudige robot te programmeren.",
        "Wat doet een egel als het vriest, en waar gaan vogels naartoe? In dit boek lees je hoe dieren de winter doorkomen, van de winterslaap tot de lange reis naar het zuiden. Met foto's, weetjes en tips om dieren te helpen in je eigen tuin.",
        "In het stadje woont een jonge draak die per ongeluk brandweerman wordt. Hij blust branden met zijn adem, maar kan zelf geen vuur meer maken. Een grappig verhaal over een draak die zijn plek zoekt.",
        "Tijdens het schoolkamp raakt Mila haar beste vriendin kwijt aan de nieuwe leerling. Ze voelt zich buitengesloten en heeft heimwee naar huis. Een verhaal over kamp, vriendschap en jaloezie.",
        "Op een eiland staat een vuurtoren waar al jaren niemand meer woont. Maar elke nacht brandt er licht. Twee vrienden gaan op onderzoek uit en ontdekken een geheim dat het eiland bijna verloor.",
        "Van pannenkoeken tot taart: twintig recepten om zelf te maken, met uitleg stap voor stap. In dit boek leer je hoe je deeg maakt, hoe je proeft en wat je doet als iets mislukt.",
        "Waar leven wolven en waarom keren ze terug naar Nederland? In dit boek lees je hoe wolven leven, hoe ze jagen en hoe mensen en wolven kunnen samenleven. Met verhalen van boswachters en boeren.",
        "Een klas gaat op schoolreis naar de bergen, maar een sneeuwstorm verandert alles. Samen met de juf moet Daan met zijn klasgenoten een nacht overleven in een oude hut. Een spannend verhaal over samenwerken en over wat vriendschap waard is.",
        "Tijdschrift over wetenschap en techniek voor jongeren, met dit nummer een special over robots en ruimtevaart.",
        "Natuurtijdschrift met reportages over dieren in de winter, wolven in Europa en bijen in de stad.",
    ],
    "trefwoorden": [
        "draak; school; vriendschap", "ruimte; astronaut; ruimtestation",
        "draak; bos; broers; geheim", "verhuizen; vriendschap; scheiding",
        "school; krant; vriendschap", "vulkaan; lava; onderzoek", "wolf; bergen; dieren",
        "robot; techniek; broers", "eiland; zomer; schat", "game; programmeren; computer",
        "schat; piraten; schoolreis", "bijen; natuur; tuin", "molen; geheim; dorp",
        "heimwee; kamp; tips", "draak; vriendschap; humor", "mars; ruimte; robots",
        "voetbal; sport; vriendschap", "koken; recepten", "hardlopen; sport; druk",
        "zee; water; proefjes", "sterren; telescoop; ruimte", "ruimte; astronaut; training",
        "robot; computer; toekomst", "dieren; winter; natuur", "draak; brandweer; humor",
        "kamp; vriendschap; jaloezie", "vuurtoren; eiland; geheim", "bakken; recepten; koken",
        "wolf; natuur; nederland", "schoolreis; bergen; sneeuw", "wetenschap; robots; ruimte",
        "dieren; winter; wolf; bijen",
    ],
    "genre": [
        "fantasy", "informatief", "", "realistisch", "realistisch", "informatief", "realistisch",
        "humor", "", "informatief", "avontuur", "informatief", "spanning", "informatief", "humor",
        "informatief", "realistisch", "informatief", "", "informatief", "informatief", "avontuur",
        "informatief", "informatief", "humor", "", "spanning", "informatief", "informatief", "",
        "informatief", "",
    ],
    "aantal_paginas": [
        212, 96, 180, 156, 168, 88, 204, 144, 230, 120, 260, 72, 196, 96, 132, 104, 176, 128, 188,
        80, 112, 160, 96, 88, 124, 152, 172, 64, 100, 208, 64, 72,
    ],
    "Gemiddelde beoordeling": [
        4, 3, None, 2, 3, None, 4, 3, None, 3, 4, None, 3, None, 4, 3, None, 2, None, 3, None, 4,
        None, 3, None, 2, 3, None, 4, None, None, 3,
    ],
})


In [34]:
def load_books():
    df = raw_books.copy()
    return df                                      

books = load_books()
titles = books["Titel"].tolist()   # van titel naar rijnummer, voor similar_items
books[["Titel", "genre", "Korte omschrijving"]].head(8)


,Titel,genre,Korte omschrijving
0,de draak onder de school,fantasy,Een meisje ontdekt dat de oude draak onder haa...
1,leven in de ruimte,informatief,"Hoe eten, slapen en douchen astronauten aan bo..."
2,het drakenei,,Twee broers vinden in het bos een enorm ei dat...
3,nieuw in de stad,realistisch,Na de scheiding van haar ouders verhuist Noor ...
4,code rood op de redactie,realistisch,Op de schoolkrant lekt een geheim uit. Wie hee...
5,vulkanen van dichtbij,informatief,Waarom barsten vulkanen uit en hoe meten onder...
6,de laatste wolf,realistisch,In een dorp in de bergen wordt de laatste wolf...
7,mijn broer de robot,humor,Sems ouders kopen een robot die alles beter ka...


## 2 · `build_vectors`: elk blok normaliseren, wegen, naast elkaar

Hetzelfde `tokenize_nostop` als in de vorige les. Daarna:

1. **tekstblok**: `TfidfVectorizer` op `Korte omschrijving`; rijen hebben al lengte 1.
2. **genreblok**: `CountVectorizer(binary=True)` op `genre`; de standaardtokenizer splitst op `;` omdat dat geen letter is, en geen genre bevat een spatie. Dat geeft één kolom per genre met een 1 als het genre erbij hoort: dezelfde one-hot als `MultiLabelBinarizer`, maar hij werkt direct op de tekstkolom. Rijen normaliseren, anders weegt een titel met twee genres zwaarder dan één met één genre.
3. elk blok maal zijn gewicht, en naast elkaar met `scipy.sparse.hstack`.

Een tekstblok en een categorieblok zien er zo hetzelfde uit: kolom in, vectorizer erop, normaliseren, wegen. Daardoor wordt `build_vectors_full` een lus over de kolommen van jullie rollentabel.

Dat is **manier 2** uit de tabel: apart vectoriseren, dan samenvoegen, één cosine over het geheel.


In [35]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)          # eenmalig, daarna uit de lokale cache

stop_words = set(stopwords.words("dutch"))
print("NLTK Nederlandse stopwoorden:", len(stop_words))
print(sorted(stop_words)[:40], "...")

EIGEN_STOPWOORDEN = {"boek", "verhaal", "jongen", "meisje", "jaar"}
STOPWORDS = stop_words | EIGEN_STOPWOORDEN

def tokenize_nostop(text):
    return [t for t in re.findall(r"\w+", str(text).lower()) if t not in STOPWORDS]

NLTK Nederlandse stopwoorden: 101
['aan', 'al', 'alles', 'als', 'altijd', 'andere', 'ben', 'bij', 'daar', 'dan', 'dat', 'de', 'der', 'deze', 'die', 'dit', 'doch', 'doen', 'door', 'dus', 'een', 'eens', 'en', 'er', 'ge', 'geen', 'geweest', 'haar', 'had', 'heb', 'hebben', 'heeft', 'hem', 'het', 'hier', 'hij', 'hoe', 'hun', 'iemand', 'iets'] ...


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

weights = {"Korte omschrijving": 1.0, "genre": 0.3}


def build_vectors(df, weights, verbose=True, return_columns=False):
    """One sparse row per title: [w_tekst * tfidf(Korte omschrijving) | w_genre * onehot(genre)].
    Met return_columns=True komt er ook een lijst kolomnamen mee (eerst de woorden, dan de genres),
    zodat je de matrix leesbaar kunt printen."""
    blocks = []

    # --- Blok 1: omschrijving als TF-IDF -------------------------------------------------
    tfidf = TfidfVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False)
    blocks.append(weights.get("Korte omschrijving", 1.0) * tfidf.fit_transform(df["Korte omschrijving"]))

    # --- Blok 2: genre als one-hot -------------------------------------------------------
    # CountVectorizer met de standaardtokenizer: die splitst op alles wat geen letter of cijfer is,
    # dus ook op ";". Geen enkel genre bevat een spatie, dus dat is genoeg: één kolom per genre.
    # binary=True: een genre telt 1 of 0, nooit 2 (anders zou "humor; humor" dubbel tellen).
    # Dit is precies wat MultiLabelBinarizer doet, maar dan rechtstreeks op de tekstkolom,
    # zodat elke kolom uit de rollentabel op dezelfde manier een blok wordt.
    onehot = CountVectorizer(binary=True)
    genre = onehot.fit_transform(df["genre"]).astype(float)
    # normalize: rij op lengte 1. CountVectorizer doet dat niet zelf; zonder deze stap
    # weegt een titel met twee genres sqrt(2) keer zo zwaar als een titel met één genre (zie deel 4).
    # Een titel zónder genre blijft een rij van nullen: normalize laat die gewoon staan.
    blocks.append(weights.get("genre", 0.0) * normalize(genre))

    # --- Samenvoegen ---------------------------------------------------------------------
    # hstack zet de blokken naast elkaar: [ woordkolommen | genrekolommen ].
    # Omdat elk blok lengte 1 heeft, bepaalt alleen het gewicht hoe zwaar het meetelt in de cosine.
    X = hstack(blocks).tocsr()
    # Kolomnamen van het geheel: de woorden van het tekstblok, daarna de genres met prefix "genre=".
    columns = list(tfidf.get_feature_names_out()) + [f"genre={g}" for g in onehot.get_feature_names_out()]
    if verbose:
        print(f"tekstblok: {blocks[0].shape[1]} kolommen · genreblok: {blocks[1].shape[1]} kolommen ({list(onehot.get_feature_names_out())})")
    return (X, columns) if return_columns else X


X, COLUMNS = build_vectors(books, weights, return_columns=True)
X.shape


tekstblok: 390 kolommen · genreblok: 6 kolommen (['avontuur', 'fantasy', 'humor', 'informatief', 'realistisch', 'spanning'])


(32, 396)

In [37]:
# --- Hoe vol is X? ---------------------------------------------------------------------
# X is sparse: alleen cellen die niet nul zijn worden opgeslagen. Dit is de hele matrix in één regel.
n_rows, n_cols = X.shape
print(f"X: {n_rows} titels x {n_cols} kolommen, {X.nnz} cellen gevuld ({100 * X.nnz / (n_rows * n_cols):.1f}%)")

# --- Drie titels naast elkaar: tekstblok boven, genreblok onder -----------------------------
# Per titel de zwaarste woorden uit het tekstblok, plus alle genrekolommen. Nullen laten we leeg,
# dan springen de gevulde cellen eruit. Let op: het genreblok bevat precies het gewicht (0.3),
# en een titel zonder genre heeft daar niets staan.
def show_vectors(title_list, X, columns, n_words=5):
    rows = [titles.index(t) for t in title_list]
    dense = pd.DataFrame(X[rows].toarray(), index=title_list, columns=columns)
    genre_cols = [c for c in columns if c.startswith("genre=")]
    word_cols = []
    for t in title_list:                                   # top-n woorden per titel, zonder dubbelen
        top = dense.loc[t].drop(genre_cols).nlargest(n_words).index
        word_cols += [w for w in top if w not in word_cols]
    table = dense[word_cols + genre_cols].round(2).T      # features als rijen, titels als kolommen
    return table.replace(0, "")

# Zeven titels, verspreid over de genres (en één zonder genre). Vier woorden per titel houdt de tabel leesbaar.
demo = ["de draak onder de school", "de draak van de brandweer", "het drakenei", "leven in de ruimte",
        "de ruimtereis van juul", "het geheim van de oude molen", "de voetbalschool"]
show_vectors(demo, X, COLUMNS, n_words=4)


X: 32 titels x 396 kolommen, 590 cellen gevuld (4.7%)


,de draak onder de school,de draak van de brandweer,het drakenei,leven in de ruimte,de ruimtereis van juul,het geheim van de oude molen,de voetbalschool
draak,0.49,0.43,,,,,
school,0.43,,,,,,0.17
alleen,0.2,,,,,,
dood,0.2,,,,,,
adem,,0.26,,,,,
blust,,0.26,,,,,
branden,,0.26,,,,,
broers,,,0.35,,,,
geheim,,,0.28,,,0.16,
aanvoelt,,,0.2,,,,


In [16]:
# --- Lengte per blok ------------------------------------------------------------------------
# Het tekstblok heeft lengte 1 (TF-IDF normaliseert), het genreblok precies het gewicht,
# en de hele rij dus sqrt(1 + 0.3^2) = 1.04. Zonder genre blijft de rij lengte 1 houden.
# Dit is de check dat het gewicht echt bepaalt hoe zwaar een blok meetelt, en niet de vorm van de data.
n_text = sum(not c.startswith("genre=") for c in COLUMNS)
rows = [titles.index(t) for t in demo]
def row_length(M): return np.sqrt(np.asarray(M.multiply(M).sum(axis=1)).ravel())
pd.DataFrame({"tekstblok": row_length(X[rows, :n_text]),
              "genreblok": row_length(X[rows, n_text:]),
              "hele rij": row_length(X[rows])}, index=demo).round(2)


,tekstblok,genreblok,hele rij
de draak onder de school,1.0,0.3,1.04
de draak van de brandweer,1.0,0.3,1.04
het drakenei,1.0,0.0,1.00
leven in de ruimte,1.0,0.3,1.04
de ruimtereis van juul,1.0,0.3,1.04
het geheim van de oude molen,1.0,0.3,1.04
de voetbalschool,1.0,0.3,1.04


In [38]:
# Het genreblok apart: de laatste zes kolommen van X, voor de eerste acht titels.
# Elke titel met een genre heeft daar precies één cel met de waarde van het gewicht (0.3);
# "het drakenei" heeft geen genre en dus een rij van nullen. Daarom doet het gewicht daar niets.
genre_cols = [c for c in COLUMNS if c.startswith("genre=")]
pd.DataFrame(X[:8, -len(genre_cols):].toarray(), index=titles[:8], columns=genre_cols)


,genre=avontuur,genre=fantasy,genre=humor,genre=informatief,genre=realistisch,genre=spanning
de draak onder de school,0.0,0.3,0.0,0.0,0.0,0.0
leven in de ruimte,0.0,0.0,0.0,0.3,0.0,0.0
het drakenei,0.0,0.0,0.0,0.0,0.0,0.0
nieuw in de stad,0.0,0.0,0.0,0.0,0.3,0.0
code rood op de redactie,0.0,0.0,0.0,0.0,0.3,0.0
vulkanen van dichtbij,0.0,0.0,0.0,0.3,0.0,0.0
de laatste wolf,0.0,0.0,0.0,0.0,0.3,0.0
mijn broer de robot,0.0,0.0,0.3,0.0,0.0,0.0


## 3 · `similar_items`, en wat het gewicht doet

`similar_items` is `top5` uit de vorige les, maar nu op de samengevoegde matrix. Dezelfde titel drie keer: gewicht 0 (alleen tekst, dus les 1), 0.3 en 1.0. Let op: een titel zónder genre (zoals "het drakenei" in deze set) verandert nauwelijks, wat het gewicht ook is. Dat is meteen het argument om de lege genres aan te vullen, of juist niet.

In [39]:
def similar_items(title, X, n=5):
    """Top-n neighbours of a title on cosine similarity over the combined vector."""
    i = titles.index(title)
    sim = cosine_similarity(X[i], X).ravel()
    order = [j for j in np.argsort(-sim) if j != i][:n]
    return pd.DataFrame({"titel": [titles[j] for j in order], "genre": books.loc[order, "genre"].values, "cosine": sim[order].round(2)})

similar_items("de draak onder de school", X)

,titel,genre,cosine
0,de draak van de brandweer,humor,0.22
1,de voetbalschool,realistisch,0.13
2,draken bestaan niet,humor,0.12
3,de schoolreis naar de bergen,,0.09
4,nieuw in de stad,realistisch,0.08


In [40]:
def compare_weights(title, genre_weights=(0.0, 0.3, 1.0), n=5):
    out = {}
    for w in genre_weights:
        Xw = build_vectors(books, {"Korte omschrijving": 1.0, "genre": w}, verbose=False)
        out[f"w_genre={w}"] = similar_items(title, Xw, n)["titel"].values
    return pd.DataFrame(out)

compare_weights("de draak onder de school")

,w_genre=0.0,w_genre=0.3,w_genre=1.0
0,de draak van de brandweer,de draak van de brandweer,de draak van de brandweer
1,de voetbalschool,de voetbalschool,de voetbalschool
2,draken bestaan niet,draken bestaan niet,de schoolreis naar de bergen
3,de schoolreis naar de bergen,de schoolreis naar de bergen,draken bestaan niet
4,nieuw in de stad,nieuw in de stad,nieuw in de stad


In [41]:
compare_weights("leven in de ruimte")

,w_genre=0.0,w_genre=0.3,w_genre=1.0
0,de ruimtereis van juul,op zoek naar mars,op zoek naar mars
1,op zoek naar mars,waarom de zee zout is,waarom de zee zout is
2,waarom de zee zout is,eerste hulp bij heimwee,eerste hulp bij heimwee
3,eerste hulp bij heimwee,dieren in de winter,dieren in de winter
4,dieren in de winter,de ruimtereis van juul,sterren kijken voor beginners


> **Wat je ziet.** Bij 0 is dit les 1. Bij 0.3 schuiven genregenoten een plek op, maar de inhoud wint. Bij 1.0 telt genre even zwaar als vijftig woorden omschrijving: alles met hetzelfde genre staat bovenaan, ongeacht waar het over gaat. Welk gewicht je wilt is een keuze met een reden; de steekproef van tien titels in de volgende les beslist.

---
## Opdracht · Jullie eerste weging

**Doel.** Na deze opdracht geeft `similar_items(title)` voor elke titel in jullie catalogus de best passende boeken terug op basis van júllie combinatie van kolommen en gewichten, en kun je bij een boek dat er niet bij past aanwijzen welke kolom of welk woord dat veroorzaakt. Opdracht 1 ging over de dummydata; vanaf hier werk je op jullie eigen schone csv.

**Stappen**

1. **Bouwen.** Zet `build_vectors(df, weights)` in `recommender.py`: per tekstkolom uit jullie rollentabel een TF-IDF-blok, per categoriekolom een one-hot-blok, genormaliseerd, maal gewicht, `hstack`. Begin met twee blokken (Korte omschrijving en genre) en breid daarna uit naar álle blokken uit de rollentabel. Bouw `similar_items(title, n)` op de gelijkenismatrix van de hele catalogus; `recommender.py` leest de schone csv en werkt op elke titel.
2. **Startgewichten vastleggen, vóórdat je draait.** Eén regel per blok in de gewichtentabel hieronder: welk gewicht, en waarom. Waarom is de ene kolom belangrijker dan de andere? Wat verwacht je als dit blok zwaarder wordt?
3. **Experimenteren.** Varieer per experiment één gewicht en zet de top-n's naast elkaar, zoals `compare_weights` hierboven doet. Zelf kiezen welke waarden je probeert; wel steeds één ding tegelijk veranderen. Eerst met drie vaste titels en top 5, daarna met een steekproef van tien titels, verspreid over genres en niveaus, en top 10 (`sample_check` in het startpunt hieronder).
4. **Bespreken en bijstellen.** Beoordeel de best passende boeken met goed, twijfel of fout. Is dit wat je wilt? Zo niet: pas het gewicht aan en draai opnieuw. Elk experiment komt in het log, ook de experimenten die je terugdraait.

**Gewichtentabel (invullen vóór het eerste experiment):**

| Blok (kolom) | Startgewicht | Waarom dit gewicht, en waarom zwaarder of lichter dan de andere blokken | Verwachting als het omhoog gaat |
|---|---|---|---|
| Korte omschrijving |  |  |  |
| genre |  |  |  |
| ... |  |  |  |

**Experimentlog (één rij per experiment):**

| # | Wat veranderd (blok, van → naar) | Titel(s) | Top-n vóór | Top-n na | Oordeel (beter / slechter / gelijk) en waarom | Houden? |
|---|---|---|---|---|---|---|
| 1 |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |

**Afhebben voor de volgende les**

- [ ] De rollentabel compleet (sjabloon in `logboek.md`), met per kolom een reden en één cijfer uit jullie EDA dat de keuze onderbouwt; bij `Niveau`, `doelgroep`, `beschikbaar_formaat`, `publicatiejaar`, `aantal_paginas` en `Gemiddelde beoordeling` ook het alternatief dat je niet koos.
- [ ] `recommender.py` leest de schone csv; `build_vectors` met álle blokken uit de rollentabel; `similar_items` werkt op elke titel uit jullie catalogus.
- [ ] De gewichtentabel, ingevuld vóór het eerste experiment.
- [ ] Het experimentlog, inclusief wat je terugdraaide en waarom.
- [ ] De steekproef: tien titels, de best passende boeken beoordeeld met goed, twijfel of fout; gewichten aangepast en opnieuw gedraaid.
- [ ] Keuzelog: combineermanier 1, 2 of 3 en waarom niet de andere; eindgewichten met het verschil ten opzichte van de start.
- [ ] *Optioneel:* `trefwoorden` als eigen one-hot-blok naast de TF-IDF op `Korte omschrijving` (zie `VECTORIZERS` hieronder). Wordt de steekproef beter? Noteer gewicht én score.

*Vragen.* Zet het gewicht van `genre` op 0 en daarna op jullie hoogste waarde: wat verandert er in de top 10 van dezelfde titel? Zoek één titel die daardoor beter werd en één die slechter werd, met de reden (welk woord of welk genre). Welke kolom kun je weghalen zonder dat de steekproef slechter wordt, en wat zegt dat? Zoek een titel waarvan de best passende boeken fout zijn: welke kolom of welk woord zit daarachter? Welke keuze uit de rollentabel draaien jullie nu al terug?

*Klaar als.* `similar_items` werkt op elke titel uit de catalogus; de gewichtentabel, het experimentlog en de steekproeftabel met tien titels staan in het notebook; de gewichten en de reflectie staan in het keuzelog.


In [21]:
# Startpunt voor recommender.py: breid build_vectors uit met de blokken uit jullie rollentabel.

# Per kolom uit de rollentabel: welke vectorizer hoort erbij?
#   tekstkolom      -> TfidfVectorizer met tokenize_nostop
#   categoriekolom  -> CountVectorizer(binary=True): one-hot, meerdere labels per cel gescheiden door ";" mag
# Vul aan met de kolommen uit jullie eigen csv.
VECTORIZERS = {
    "Korte omschrijving": lambda: TfidfVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False),
    "genre":       lambda: CountVectorizer(binary=True),
    "trefwoorden": lambda: CountVectorizer(binary=True),
}

def build_vectors_full(df, weights):
    """weights: dict with one entry per block, e.g.
    {"Korte omschrijving": 1.0, "trefwoorden": 0.5, "Titel": 0.2, "genre": 0.3, "Type materiaal": 0.1}
    Voor elke kolom in weights: VECTORIZERS[kolom]() -> fit_transform -> normalize -> maal gewicht; dan hstack."""
    raise NotImplementedError

def sample_check(title_list, X, n=5):
    """Return a DataFrame with one row per (title, neighbour) and an empty 'oordeel' column
    to fill in by hand: goed / twijfel / fout."""
    raise NotImplementedError
